# 06c — Final model: Siamese U-Net (concat), Focal + Dice, 40 epochs

### 1. setup

In [ ]:
BRANCH = "main"

%cd /tmp
!rm -rf damage-detection
!git clone -b {BRANCH} https://github.com/azrairemd/damage-detection.git
%cd damage-detection
!pip install -q -r requirements.txt
!pip install -q -e .

### 2. imports

In [ ]:
import os
import shutil
import time

import torch
from torch.utils.data import DataLoader

from src.data.dataset import XBDDataset
from src.losses.combined_loss import CombinedLoss
from src.models.siamese_unet import SiameseUNet
from src.train import fit
from src.utils.config import load_config

### 3. configs

In [ ]:
CONFIG_PATH = "configs/siamese_concat_fd40.yaml"

config = load_config(CONFIG_PATH)
data = config["data"]
train_cfg = config["training"]
loss_cfg = config["loss"]
model_cfg = config["model"]

### 4. device

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

### 5. data

In [ ]:
train_ds = XBDDataset(data["train_split"], data["xbd_root"], data["mask_root"],
                      train=True, crop_size=data["crop_size"])
val_ds = XBDDataset(data["val_split"], data["xbd_root"], data["mask_root"], train=False)

train_loader = DataLoader(train_ds, batch_size=train_cfg["batch_size"], shuffle=True,
                          num_workers=train_cfg["num_workers"], drop_last=True)
val_loader = DataLoader(val_ds, batch_size=train_cfg["val_batch_size"], shuffle=False,
                        num_workers=train_cfg["num_workers"])

print("Train batches:", len(train_loader), " Val batches:", len(val_loader))

### 6. model

In [ ]:
model = SiameseUNet(encoder=model_cfg["encoder"], fusion=model_cfg["fusion"]).to(device)
loss_fn = CombinedLoss(alpha=torch.tensor(loss_cfg["class_weights"]),
                       gamma=loss_cfg["gamma"], lam=loss_cfg["lam"])
optimizer = torch.optim.AdamW(model.parameters(), lr=train_cfg["learning_rate"])

### 7. training

In [ ]:
out_dir = train_cfg["out_dir"]
os.makedirs(out_dir, exist_ok=True)
shutil.copy(CONFIG_PATH, os.path.join(out_dir, "config.yaml"))

start = time.time()
best = fit(model, train_loader, val_loader, loss_fn, optimizer, device,
           max_epochs=train_cfg["max_epochs"], out_dir=out_dir)

print(f"Best val score: {best:.4f}")
print(f"Total time: {(time.time() - start) / 3600:.1f} hours")